# 09 · Real tree height around every roof (Meta 1 m canopy height map)

**Run this in Google Colab.** It takes about 1–2 hours. If Colab disconnects, run it again: finished map tiles are
saved to Drive and skipped.

Why: the ESA WorldCover tree % (notebooks 03 and 05) is ~100% for most Kerala roofs, so it cannot tell a sunny roof
from a shaded one. The Meta / WRI **global canopy height map** gives tree height in metres at ~1 m pixels
(Tolan et al. 2024, CC BY 4.0, imagery mostly 2018–2020).

What it adds to every roof (pixels are grouped into 3 × 3 blocks of ~3.5 m first, which is 4× faster and
gives nearly the same numbers):
1. **`chm_roof_max`**: tallest tree within ~5 m of the roof centre (trees over or touching the roof), metres.
2. **`chm_max_15m`**: tallest tree within ~15 m, metres.
3. **`tall_pct_15m`**: % of the ground within ~15 m covered by trees 5 m or taller.
4. **`tall_pct_30m`**: the same within ~30 m.

Steps: for each of the ~20 map tiles that hold Kerala roofs (~600 MB each), download it, read it in strips, compute
the 4 numbers for the roofs on it, save them, delete the tile. The next tile downloads while one is processed.

Output: `MyDrive/rooftop_solar/data/features/canopy_features.parquet` (one row per roof id).
At the end it checks how well each feature separates good from not-good labelled roofs.

**How to run:** Runtime → Run all. Allow Google Drive access.

In [ ]:
# Step 1: connect Google Drive, set folders, copy the roofs to the fast Colab disk
from google.colab import drive
drive.mount('/content/drive')

import os, shutil, time

PROJECT = '/content/drive/MyDrive/rooftop_solar'
FEATURES = f'{PROJECT}/data/features'
LABELS = f'{PROJECT}/data/labels'
RESULTS = f'{PROJECT}/results'
LOGS = f'{PROJECT}/logs'
LOCAL = '/content/work'
PARTS = f'{FEATURES}/canopy_parts'          # one small file per finished map tile (lets a re-run skip it)
for folder in (RESULTS, LOGS, LOCAL, PARTS):
    os.makedirs(folder, exist_ok=True)

LOG_FILE = f'{LOGS}/09_log.txt'             # appended to, so a re-run after a disconnect keeps the earlier lines


def log(*parts):
    """Print, and also save to Google Drive so results survive if the tab closes."""
    text = ' '.join(str(p) for p in parts)
    print(text)
    with open(LOG_FILE, 'a') as f:
        f.write(text + '\n')


def log_table(pdf, csv_name=None):
    """Log a small pandas table; optionally save it as a CSV in results/."""
    log(pdf.to_string(index=False))
    if csv_name:
        pdf.to_csv(f'{RESULTS}/{csv_name}', index=False)


T_START = time.time()
log(f'\n===== run started {time.strftime("%Y-%m-%d %H:%M")} =====')
shutil.copytree(f'{FEATURES}/roof_features_v2', f'{LOCAL}/roof_features_v2', dirs_exist_ok=True)
log(f'Roofs copied to {LOCAL} in {time.time() - T_START:.0f} s')

In [ ]:
# Step 2: install DuckDB and rasterio (NumPy, pandas and SciPy are already in Colab)
!pip install -q duckdb==2.0.0 rasterio

import subprocess
import duckdb
import numpy as np
import pandas as pd
import rasterio
from rasterio.windows import Window
from scipy.ndimage import maximum_filter, uniform_filter

con = duckdb.connect(f'{LOCAL}/canopy.duckdb')
con.sql("INSTALL spatial; LOAD spatial;")
con.sql("SET geometry_always_xy = true; SET memory_limit = '4GB';")   # leave RAM for the NumPy steps

In [ ]:
# Step 3: load every roof centre and find which map tile it falls on
t = time.time()
con.execute(f"""
    CREATE OR REPLACE TABLE roofs AS
    SELECT row_number() OVER (ORDER BY id) AS rid, id, district, centre_lon, centre_lat   -- same rid on every run
    FROM read_parquet('{LOCAL}/roof_features_v2/*/*.parquet', hive_partitioning = true)
""").fetchall()
xy = con.execute('SELECT rid, centre_lon, centre_lat FROM roofs ORDER BY rid').fetchnumpy()
rid = np.asarray(xy['rid'], dtype=np.int64)
lon = np.asarray(xy['centre_lon'], dtype='float64')
lat = np.asarray(xy['centre_lat'], dtype='float64')

# The map is cut into zoom-9 web-map tiles (EPSG:3857), named by their quadkey
R = 6378137.0
mx = R * np.radians(lon)                                         # web-Mercator metres
my = R * np.log(np.tan(np.pi / 4 + np.radians(lat) / 2))
N = 2 ** 9


def tile_xy(lon, lat):
    return (((lon + 180) / 360 * N).astype(np.int64),
            ((1 - np.log(np.tan(np.radians(lat)) + 1 / np.cos(np.radians(lat))) / np.pi) / 2 * N).astype(np.int64))


def quadkey(x, y, z=9):
    return ''.join(str(((x >> (z - 1 - i)) & 1) + 2 * ((y >> (z - 1 - i)) & 1)) for i in range(z))


kx, ky = tile_xy(np.array([76.3]), np.array([9.95]))
assert quadkey(int(kx[0]), int(ky[0])) == '123321102'           # Kochi is on tile 123321102
tx, ty = tile_xy(lon, lat)
key = tx * N + ty
order = np.argsort(key, kind='stable')
uniq, starts = np.unique(key[order], return_index=True)
TILES = {quadkey(int(k // N), int(k % N)): order[s:e]
         for k, s, e in zip(uniq, starts, list(starts[1:]) + [len(order)])}
del order, key, tx, ty, lon, lat
log(f'{len(rid):,} roofs on {len(TILES)} map tiles ({time.time() - t:.0f} s)')

In [ ]:
# Step 4: download each tile, compute the 4 shade numbers for its roofs, save, delete the tile
URL = 'https://dataforgood-fb-data.s3.amazonaws.com/forests/v1/alsgedi_global_v6_float/chm/{}.tif'
TALL_M = 5           # a "tall" tree is 5 m or more (most Kerala houses are 3-8 m high)
STRIP = 2049         # tile rows read at a time (a multiple of 3)
PAD = 27             # extra pixels read around each strip, so the ~30 m windows are complete
COLS = ['chm_roof_max', 'chm_max_15m', 'tall_pct_15m', 'tall_pct_30m']


def tile_features(path, idx):
    """The 4 features for the roofs idx (positions in rid) on one tile."""
    with rasterio.open(path) as src:
        T = src.transform
        rows = np.floor((my[idx] - T.f) / T.e).astype(np.int64).clip(0, src.height - 1)
        cols = np.floor((mx[idx] - T.c) / T.a).astype(np.int64).clip(0, src.width - 1)
        o = np.argsort(rows, kind='stable')
        idx, rows, cols = idx[o], rows[o], cols[o]
        out = np.zeros((len(idx), 4), dtype=np.float32)
        for b0 in range(int(rows[0]) // STRIP * STRIP, int(rows[-1]) + 1, STRIP):
            i0, i1 = np.searchsorted(rows, [b0, b0 + STRIP])
            if i0 == i1:
                continue                                         # no roofs in this strip (sea, forest, other state)
            c = cols[i0:i1]
            top = max(b0 - PAD, 0)
            left = max((int(c.min()) - PAD) // 3 * 3, 0)
            bottom = min(b0 + STRIP + PAD, src.height)
            right = min(int(c.max()) + 1 + PAD, src.width)
            chm = src.read(1, window=Window(left, top, right - left, bottom - top))   # tree height, whole metres
            h, w = chm.shape[0] // 3 * 3, chm.shape[1] // 3 * 3
            blk = chm[:h, :w].reshape(h // 3, 3, w // 3, 3)     # 3 x 3 pixel blocks (~3.5 m)
            bmax = blk.max(axis=(1, 3))
            btall = (blk >= TALL_M).mean(axis=(1, 3), dtype=np.float32)
            del chm, blk
            br = np.minimum((rows[i0:i1] - top) // 3, bmax.shape[0] - 1)
            bc = np.minimum((c - left) // 3, bmax.shape[1] - 1)
            out[i0:i1, 0] = maximum_filter(bmax, size=3)[br, bc]          # ~10 m square
            out[i0:i1, 1] = maximum_filter(bmax, size=9)[br, bc]          # ~32 m square
            out[i0:i1, 2] = 100 * uniform_filter(btall, size=9)[br, bc]
            out[i0:i1, 3] = 100 * uniform_filter(btall, size=17)[br, bc]  # ~60 m square
    # shortcut: windows are cut at tile edges (a few roofs within 30 m of an edge lose part of their window)
    df = pd.DataFrame(np.round(out).astype(np.uint8), columns=COLS)
    df.insert(0, 'rid', rid[idx])
    return df


def fetch(tile):
    return subprocess.Popen(['curl', '-sSfL', '--retry', '5', '-o', f'{LOCAL}/{tile}.tif', URL.format(tile)])


todo = [k for k in TILES if not os.path.exists(f'{PARTS}/{k}.parquet')]
log(f'Tiles to do: {len(todo)} of {len(TILES)} (the rest were finished in an earlier run)')
t_all = time.time()
nxt = fetch(todo[0]) if todo else None
for i, tile in enumerate(todo):
    t = time.time()
    if nxt.wait() != 0:
        raise SystemExit(f'Download of tile {tile} failed. Run the notebook again: finished tiles are kept.')
    t_dl = time.time() - t
    nxt = fetch(todo[i + 1]) if i + 1 < len(todo) else None      # download the next tile in the background
    t = time.time()
    part = tile_features(f'{LOCAL}/{tile}.tif', TILES[tile])
    part.to_parquet(f'{PARTS}/{tile}.parquet.tmp', index=False)
    os.replace(f'{PARTS}/{tile}.parquet.tmp', f'{PARTS}/{tile}.parquet')   # a half-written file is never kept
    os.remove(f'{LOCAL}/{tile}.tif')
    log(f'  tile {i + 1}/{len(todo)} {tile}: {len(part):,} roofs | waited {t_dl:.0f} s for download, '
        f'computed in {time.time() - t:.0f} s')
T_CANOPY = time.time() - t_all
log(f'All tiles done in {T_CANOPY / 60:.1f} min')

In [ ]:
# Step 5: join the features to the roof ids and save one file
t = time.time()
con.execute(f"""
    COPY (
        SELECT r.id, c.chm_roof_max, c.chm_max_15m, c.tall_pct_15m, c.tall_pct_30m
        FROM roofs r JOIN read_parquet('{PARTS}/*.parquet') c USING (rid)
    ) TO '{LOCAL}/canopy_features.parquet' (FORMAT parquet, COMPRESSION zstd)
""").fetchall()
shutil.copy(f'{LOCAL}/canopy_features.parquet', f'{FEATURES}/canopy_features.parquet')
log(f'Saved {FEATURES}/canopy_features.parquet in {time.time() - t:.0f} s')

In [ ]:
# Step 6: checks and summaries (send these to Claude)
CF = f"read_parquet('{LOCAL}/canopy_features.parquet')"
V2 = f"read_parquet('{LOCAL}/roof_features_v2/*/*.parquet', hive_partitioning = true)"

log('\nCHECKS (roofs_with_canopy = all_roofs and distinct_ids = roofs_with_canopy is ideal)')
log_table(con.sql(f"""
    SELECT (SELECT count(*) FROM roofs) AS all_roofs, count(*) AS roofs_with_canopy, count(DISTINCT id) AS distinct_ids
    FROM {CF}
""").df())

log('\nCANOPY FEATURES: share of roofs at 0 and 5th / 50th / 95th percentile')
log_table(con.sql(" UNION ALL ".join(f"""
    SELECT '{c}' AS feature, round(100 * avg(CAST({c} = 0 AS DOUBLE)), 1) AS pct_zero,
           approx_quantile({c}, [0.05, 0.5, 0.95]) AS p5_p50_p95 FROM {CF}""" for c in COLS)).df())

log('\nBY DISTRICT')
log_table(con.sql(f"""
    SELECT r.district, count(*) AS roofs,
           round(avg(c.chm_roof_max), 1) AS avg_roof_max_m, round(avg(c.chm_max_15m), 1) AS avg_max_15m_m,
           round(avg(c.tall_pct_15m), 1) AS avg_tall_15m_pct, round(avg(c.tall_pct_30m), 1) AS avg_tall_30m_pct
    FROM roofs r JOIN {CF} c USING (id) GROUP BY 1 ORDER BY roofs DESC
""").df(), 'district_canopy_features.csv')


def auc(y, s):
    """Chance that a good roof has a higher value than a not-good one (0.5 = no signal; below 0.5 = good roofs lower)."""
    y, ranks = np.asarray(y, dtype=int), pd.Series(s).rank().to_numpy()
    n1, n0 = int(y.sum()), int(len(y) - y.sum())
    return round((ranks[y == 1].sum() - n1 * (n1 + 1) / 2) / (n1 * n0), 3)


# Do the features separate good roofs from not-good ones? (WorldCover features shown for comparison)
con.execute(f"CREATE OR REPLACE TABLE hand AS SELECT id, label FROM ST_Read('{LABELS}/roofs_to_label.gpkg')")
con.execute(f"CREATE OR REPLACE TABLE ai AS SELECT id, ai_label FROM ST_Read('{LABELS}/roofs_to_label_ai.gpkg')")
lab = con.execute(f"""
    SELECT h.id, h.label, a.ai_label, v.tree_pct AS tree_28m, v.tree_pct_140m AS tree_140m, v.tree_pct_300m AS tree_300m,
           c.chm_roof_max, c.chm_max_15m, c.tall_pct_15m, c.tall_pct_30m
    FROM hand h JOIN ai a USING (id) JOIN {V2} v USING (id) JOIN {CF} c USING (id)
""").df()
sets = {'your labels': lab['label'], 'AI labels': lab['ai_label']}
if os.path.exists(f'{LABELS}/adjudication.csv'):
    adj = pd.read_csv(f'{LABELS}/adjudication.csv', dtype={'id': str})
    agree = lab['label'].notna() & lab['ai_label'].notna() & ((lab['label'] == 1) == (lab['ai_label'] == 1))
    sets['gold labels'] = pd.Series(np.where(agree, lab['label'], lab['id'].map(dict(zip(adj['id'], adj['verdict'])))))
rows = []
for f in ['tree_28m', 'tree_140m', 'tree_300m'] + COLS:
    row = {'feature': f}
    for name, y in sets.items():
        ok = y.notna() & lab[f].notna()
        row[f'AUC {name}'] = auc(y[ok] == 1, lab.loc[ok, f])
    rows.append(row)
log('\nHOW WELL EACH FEATURE ALONE SEPARATES GOOD FROM NOT-GOOD ROOFS (0.5 = no signal; far from 0.5 = useful)')
log_table(pd.DataFrame(rows), 'canopy_feature_signal.csv')
log(f'\nTotal time {(time.time() - T_START) / 60:.1f} min')
log('ALL DONE')

## Done

Send Claude the text of `MyDrive/rooftop_solar/logs/09_log.txt`. Then run notebook 10.